# Dino Slayer — Notebook 3: Spatial Analysis and Decision Support

**Stage 5 of the geospatial pipeline: predictions → actionable decisions.**

A ranked list is not a plan. The number-one settlement by DIPI may sit in the far east and
number two in the far west, so acting on the ranking alone means crossing the state twice and
paying for each community's infrastructure as if it stood alone.

This notebook turns the ranking and the model into things a planner can act on:

| Section | Question answered | Technique |
|---|---|---|
| **1. Deployment bundles** | which communities should be built together? | HDBSCAN clustering + minimum spanning tree |
| **2. Where to measure next** | which unmeasured settlements are worth surveying? | model disagreement × decision relevance |
| **3. Survey routing** | in what order should a field team visit them? | greedy nearest-neighbour + 2-opt |
| **4. Line-of-sight screen** | can a mast actually reach a village? | terrain profile sampling |
| **5. Nighttime lights** | does the site have electricity at all? | VIIRS annual composites + Mann-Kendall |

**Spatial clustering** is one of the three primary machine-learning tasks in **Module 3,
Session 2.2** — unsupervised, grouping similar data points, no labels. Sections 3 and 4 are
classical geometry and graph optimisation rather than machine learning, and are labelled as
such rather than dressed up.

---

### Position in the pipeline — this notebook runs in TWO passes

Sections 4 and 5 sit either side of [`tower_los.py`](https://github.com/RextonRZ/dino-slayer/blob/main/dataset/tower_los.py):

    tower_scenarios.py       →  tower_pairs.csv
            ↓
    S4 line-of-sight screen  →  tower_pairs_los.csv
            ↓
    tower_los.py             →  tower_isolated.csv
            ↓
    S5 VIIRS join            →  tower_isolated_power.csv

`tower_isolated.csv` does **not** exist when this notebook starts — S4 produces what creates it.

| Input | Produced by | Needed for |
|---|---|---|
| `training_table.csv` | [`export_training_table.py`](https://github.com/RextonRZ/dino-slayer/blob/main/dataset/export_training_table.py) | S1 clustering |
| `model_predictions_v1.csv` | notebook 02 | S2 survey priority |
| `tower_pairs.csv` | [`tower_scenarios.py`](https://github.com/RextonRZ/dino-slayer/blob/main/dataset/tower_scenarios.py) | S4 line-of-sight |
| `tower_isolated.csv` | [`tower_los.py`](https://github.com/RextonRZ/dino-slayer/blob/main/dataset/tower_los.py), after S4 | S5 VIIRS join |

**Extra setup:** S4 rebuilds a 419 MB elevation raster if it is not already cached in Drive.
S5 requires Google Earth Engine authentication and a Cloud project id.

In [1]:
# ══════════════════════════════════════════════════════════════════════
# SETUP
# ══════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, json, os
D = "/content/drive/MyDrive/Dataset/"

df = pd.read_csv(D + "training_table.csv")
print(f"{len(df)} settlements, {df.district.nunique()} districts")

# Earth radius in km. NOTE: deliberately named R_KM, not R — a bare `R`
# is easy to overwrite later with a loaded JSON object, after which every
# distance silently becomes nonsense.
R_KM = 6371.0088

def hav_matrix(lat1, lon1, lat2, lon2):
    """Pairwise great-circle distance in km. Inputs are degrees, arrays."""
    la1, lo1, la2, lo2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dla = la2[None,:] - la1[:,None]
    dlo = lo2[None,:] - lo1[:,None]
    a = np.sin(dla/2)**2 + np.cos(la1)[:,None]*np.cos(la2)[None,:]*np.sin(dlo/2)**2
    return 2 * R_KM * np.arcsin(np.sqrt(a))

Mounted at /content/drive
1448 settlements, 25 districts


---
## 1. Deployment bundles — clustering settlements that share one build

### The problem in plain terms

Picture ten villages strung along one road, each about 15 km from town. Costing them one at a
time charges each for its own cable all the way back: ten times fifteen kilometres of
trenching. Nobody builds that way — the cable reaching village ten already passes villages one
to nine.

A **bundle** is a group close enough that one build serves all of them.

### Why HDBSCAN rather than DBSCAN

DBSCAN was tried first and collapsed. With a single global radius it produced one cluster
holding the majority of Sabah's settlements: they form a connected chain, and single-linkage
walks the whole state. The failure was not bad luck — it is `eps` being one number when
settlements are packed near towns and scattered inland, so no single radius fits both.

**HDBSCAN** builds a cluster hierarchy and cuts it per region, which removes the "why that
radius?" question entirely.

### Two scoping decisions that matter

**Only fibre settlements are clustered.** A trench passing a village serves it, which is what
makes sharing real. Towers, satellite and community Wi-Fi are per-site builds — a mast does
not get cheaper because its neighbour has one.

**Clustering is run on the fibre subset**, not on all settlements followed by a filter.
Filtering afterwards over-merges: two fibre villages 40 km apart end up in one bundle sharing
a trench that could not exist.

The fibre rule (`backhaul_km ≤ 15` and `pop_2km ≥ 3000`) is a published constant of the
recommender, restated here so this notebook selects exactly the same settlements.

In [2]:
# ══════════════════════════════════════════════════════════════════════
# Verify backhaul_km reproduces from the raw coordinates.
# Rounding to 2 dp matters: it is the precision the dashboard uses, and a
# raw value let one extra settlement slip under a budget cap in testing.
# ══════════════════════════════════════════════════════════════════════
anchors = df[df.place.isin(["city","town"])]
print(f"{len(anchors)} anchors (cities + towns)")

Dm = hav_matrix(df.lat.values, df.lon.values, anchors.lat.values, anchors.lon.values)
df["backhaul_calc"] = np.round(Dm.min(axis=1), 2)

delta = (df.backhaul_calc - df.backhaul_km).abs()
print(f"max difference vs the supplied backhaul_km: {delta.max():.4f} km")
print(f"rows differing by more than 0.01 km      : {(delta > 0.01).sum()}")
print("→ zero differences means this notebook and the dashboard agree exactly.")

59 anchors (cities + towns)
max difference vs the supplied backhaul_km: 0.0100 km
rows differing by more than 0.01 km      : 3
→ zero differences means this notebook and the dashboard agree exactly.


In [3]:
# ══════════════════════════════════════════════════════════════════════
# HDBSCAN on the fibre-eligible settlements, then a minimum spanning tree
# per bundle (Prim's algorithm).
#
# The MST is the cheapest set of links connecting every member of a bundle.
# The root is the member closest to a town and pays the full run in from
# town; every other member is charged only the edge that attached it.
#
# A 1 km floor is applied to every edge INCLUDING the root, matching the
# costing function, so a settlement inside a town is not costed at zero.
#
# metric="haversine" requires RADIANS, and lat before lon.
# ══════════════════════════════════════════════════════════════════════
import sklearn
from sklearn.cluster import HDBSCAN
print("sklearn", sklearn.__version__, "(HDBSCAN requires >= 1.3)")

FIBRE_MAX_KM, FIBRE_MIN_POP = 15, 3000        # published recommender constants
df["is_fibre"] = (df.backhaul_km <= FIBRE_MAX_KM) & (df.pop_2km >= FIBRE_MIN_POP)
fib = df[df.is_fibre].copy()
print(f"fibre-eligible settlements: {len(fib)}")

fib["cl"] = HDBSCAN(min_cluster_size=5, metric="haversine").fit_predict(
    np.radians(fib[["lat","lon"]].values))
print(f"bundles: {fib.cl.max()+1} · unclustered (noise): {(fib.cl == -1).sum()}")

trunk = {}
for cl, grp in fib[fib.cl >= 0].groupby("cl"):
    idx  = grp.index.tolist()
    root = grp.backhaul_km.idxmin()                          # closest to a town
    trunk[root] = max(1.0, float(df.at[root, "backhaul_km"]))  # someone pays the full run in
    seen, rest = [root], set(idx) - {root}
    while rest:                                               # Prim's: attach the nearest
        dkm, j = min((hav_matrix(np.array([df.at[i,"lat"]]), np.array([df.at[i,"lon"]]),
                                 np.array([df.at[j,"lat"]]), np.array([df.at[j,"lon"]]))[0,0], j)
                     for i in seen for j in rest)
        trunk[j] = max(1.0, round(float(dkm), 2))
        seen.append(j); rest.discard(j)

# Noise points and non-fibre settlements keep their full run: with no near
# neighbour there is no trench to share. This makes the estimate conservative.
for i in df.index:
    trunk.setdefault(i, max(1.0, float(df.at[i, "backhaul_km"])))

df["trunk_km"] = pd.Series(trunk)
df["cl"] = fib.cl.reindex(df.index).fillna(-1).astype(int)

f = df[df.is_fibre]
print(f"\nfull run to town, floored : {f.backhaul_km.clip(lower=1).sum():.1f} km")
print(f"shared trunk after bundling: {f.trunk_km.sum():.1f} km")
print(f"ratio                      : {f.backhaul_km.clip(lower=1).sum()/f.trunk_km.sum():.2f}x")
print(f"settlements that get cheaper: {(f.trunk_km < f.backhaul_km.clip(lower=1)).sum()} of {len(f)}")

cross = fib[fib.cl >= 0].groupby("cl").district.nunique()
print(f"\nbundles crossing a district boundary: {(cross>1).sum()} of {len(cross)}")
print("  Fibre does not respect administrative lines. Those corridors are cheapest")
print("  ONLY IF the districts involved co-fund them — worth stating, not hiding.")

sklearn 1.6.1 (HDBSCAN requires >= 1.3)
fibre-eligible settlements: 323
bundles: 17 · unclustered (noise): 35

full run to town, floored : 1363.5 km
shared trunk after bundling: 890.0 km
ratio                      : 1.53x
settlements that get cheaper: 206 of 323

bundles crossing a district boundary: 7 of 17
  Fibre does not respect administrative lines. Those corridors are cheapest
  ONLY IF the districts involved co-fund them — worth stating, not hiding.


In [4]:
# ══════════════════════════════════════════════════════════════════════
# EXPORT clusters.json — one entry per settlement, all 1,448.
# cl = -1 means unclustered (noise or non-fibre).
#
# trunk_km here is the RAW spanning-tree edge. The shipped cost applies
# min(trunk, direct run) downstream, because a few settlements sit further
# from their bundle neighbour than from town and a planner would simply
# build the shorter one.
#
# No extra keys are written: anything iterating this file expects every
# key to be a settlement id.
# ══════════════════════════════════════════════════════════════════════
clusters = {r.settlement_id: {"cl": int(r.cl), "trunk_km": round(float(r.trunk_km), 2)}
            for r in df.itertuples()}
p = D + "outputs/clusters.json"
os.makedirs(D + "outputs", exist_ok=True)
json.dump(clusters, open(p, "w"))

print(f"saved {p} — {os.path.getsize(p)/1024:.0f} KB, {len(clusters)} entries")
assert all(k.startswith("S") for k in clusters), "non-settlement key present"
print(json.dumps(dict(list(clusters.items())[:3]), indent=2))

saved /content/drive/MyDrive/Dataset/outputs/clusters.json — 55 KB, 1448 entries
{
  "S0000": {
    "cl": 14,
    "trunk_km": 1.0
  },
  "S0004": {
    "cl": -1,
    "trunk_km": 12.32
  },
  "S0007": {
    "cl": -1,
    "trunk_km": 15.7
  }
}


### Parameter sensitivity

`min_cluster_size = 5` is a choice, so the honest question is how much the answer depends on
it. The sweep below re-runs the whole clustering and spanning-tree calculation at six values.

If the result moves sharply between neighbouring settings, the parameter is load-bearing and
the number was effectively tuned. If it sits on a plateau, the choice does not matter much and
that is what makes it defensible.

In [5]:
# ══════════════════════════════════════════════════════════════════════
# min_cluster_size SWEEP
# (HDBSCAN has no `eps`, which is why it replaced DBSCAN — so this is the
# equivalent sensitivity test.)
# ══════════════════════════════════════════════════════════════════════
base_km = f.backhaul_km.clip(lower=1).sum()
sweep_rows = []

for mcs in [3, 4, 5, 6, 8, 10]:
    lab = HDBSCAN(min_cluster_size=mcs, metric="haversine").fit_predict(
        np.radians(fib[["lat","lon"]].values))
    tmp, t = fib.assign(cl=lab), {}
    for cl, g in tmp[tmp.cl >= 0].groupby("cl"):
        root = g.backhaul_km.idxmin()
        t[root] = max(1.0, float(df.at[root, "backhaul_km"]))
        seen, rest = [root], set(g.index) - {root}
        while rest:
            dk, j = min((hav_matrix(np.array([df.at[i,"lat"]]), np.array([df.at[i,"lon"]]),
                                    np.array([df.at[j,"lat"]]), np.array([df.at[j,"lon"]]))[0,0], j)
                        for i in seen for j in rest)
            t[j] = max(1.0, round(float(dk), 2)); seen.append(j); rest.discard(j)
    for i in fib.index:
        t.setdefault(i, max(1.0, float(df.at[i, "backhaul_km"])))
    km = sum(t.values())
    sweep_rows.append({"mcs": mcs, "clusters": int(tmp.cl.max()+1),
                       "noise": int((tmp.cl == -1).sum()),
                       "trunk_km": round(km, 1), "ratio": round(base_km/km, 2)})
    print(f"mcs={mcs:2d} → {sweep_rows[-1]['clusters']:2d} bundles, "
          f"{sweep_rows[-1]['noise']:3d} noise, {km:6.1f} km, {base_km/km:.2f}x")

json.dump({"parameter":"min_cluster_size", "algorithm":"HDBSCAN haversine",
           "n_fibre": int(len(fib)), "baseline_km": round(float(base_km),1),
           "results": sweep_rows, "shipped": 5},
          open(D + "outputs/cluster_sensitivity.json", "w"), indent=2)
print("\nsaved ✓ outputs/cluster_sensitivity.json")

mcs= 3 → 40 bundles,  45 noise,  932.7 km, 1.46x
mcs= 4 → 22 bundles,  36 noise,  933.5 km, 1.46x
mcs= 5 → 17 bundles,  35 noise,  890.0 km, 1.53x
mcs= 6 → 15 bundles,  41 noise, 1022.8 km, 1.33x
mcs= 8 → 11 bundles,  59 noise,  976.9 km, 1.40x
mcs=10 → 10 bundles,  63 noise,  978.5 km, 1.39x

saved ✓ outputs/cluster_sensitivity.json


---
## 2. Where to measure next

216 settlements have no measurement. Which are worth sending a field team to?

**The naive answer is wrong.** Ranking by predicted slowness sends teams to places the model
is already confident about — measuring them confirms what is known and changes no decision.

**The right question is decisiveness:** does the estimate sit close enough to the service
threshold, relative to the model's own uncertainty, that a measurement could land on either
side of it? Those are the settlements where new evidence changes what happens next.

### The formula, and how it was arrived at

This ranking was revised repeatedly, each time because manual inspection of the top ten
showed it surfacing the wrong places:

| Version | What it produced | Why it was wrong |
|---|---|---|
| v1 | settlements never predicted ranked top | they were assigned maximum uncertainty by default |
| v2 | estimates of 141–174 Mbps ranked top | wide intervals alone qualified them |
| v3 | correlated −0.49 with model disagreement | it preferred settlements already understood |
| **v4 (shipped)** | near-threshold, populated, uncertain | — |

The final score is:

- **Decisiveness (55%)** — how close the estimate is to the threshold, in units of the model's
  own disagreement. Estimates far above the line score zero.
- **Impact (45%)** — population and nearby anchor institutions.
- **A hard gate** — settlements whose interval crosses the threshold always outrank those whose
  interval does not.
- **`pop_2km > 0` required** — a survey needs someone to serve.



In [6]:
# ══════════════════════════════════════════════════════════════════════
# MEASUREMENT PRIORITY
# Ranks WHERE TO COLLECT DATA. It is not an infrastructure recommendation.
# ══════════════════════════════════════════════════════════════════════
THRESHOLD = 21.0        # Mbps service line

preds = pd.read_csv(D + "outputs/model_predictions_v1.csv")
tt    = pd.read_csv(D + "training_table.csv")[["settlement_id","pop_2km","n_schools_3km","n_clinics_3km"]]
cand  = preds[preds.speed_source == "modelled"].merge(tt, on="settlement_id", how="left")
print(f"unmeasured settlements        : {len(cand)}")

cand = cand[cand.pop_2km > 0].copy()
print(f"with population within 2 km   : {len(cand)}")

# Does the disagreement interval cross the service line?
cand["straddles"] = (cand.prediction_lower < THRESHOLD) & (cand.prediction_upper > THRESHOLD)
print(f"straddling the {THRESHOLD:.0f} Mbps line   : {cand.straddles.sum()}")

# Closer to the line relative to our own uncertainty = more decisive to measure.
# Estimates above 3x the threshold score zero: a 95 Mbps estimate is not a
# borderline case however wide its interval.
cand["decisiveness"] = np.where(
    cand.predicted_speed > 3*THRESHOLD, 0.0,
    1 / (1 + (cand.predicted_speed - THRESHOLD).abs() / cand.prediction_std.clip(upper=15)))

imp = (np.log1p(cand.pop_2km).rank(pct=True) * 0.6 +
       (cand.n_schools_3km + cand.n_clinics_3km).rank(pct=True) * 0.4)

score = 0.55 * cand.decisiveness.rank(pct=True) + 0.45 * imp
# Hard gate: straddlers occupy 0.5-1.0, everyone else 0.0-0.5. No overlap.
cand["measurement_priority"] = np.where(cand.straddles, 0.5 + score/2, score/2).round(3)

cand["measurement_priority_reason"] = (
    "est " + cand.predicted_speed.round(1).astype(str) + " ±" +
    cand.prediction_std.round(1).astype(str) + " Mbps (" +
    cand.prediction_lower.round(1).astype(str) + "–" + cand.prediction_upper.round(1).astype(str) + ") · " +
    np.where(cand.straddles, f"crosses the {THRESHOLD:.0f} Mbps line", "clear of the line") + " · " +
    cand.pop_2km.astype(int).astype(str) + " people within 2 km")

cand["name"] = cand["name"].fillna("Unnamed (" + cand.settlement_id + ")")
cand = cand.sort_values("measurement_priority", ascending=False)
cand.to_csv(D + "outputs/measurement_priority_v1.csv", index=False)

n = int(cand.straddles.sum())
print(f"\ngate check — top {n} are all straddlers: {cand.head(n).straddles.sum()}/{n}")
print(f"correlation with model disagreement: "
      f"{cand.measurement_priority.corr(cand.prediction_std, method='spearman'):+.3f}")
print("\nTOP 10 — inspect these by eye before anyone acts on them")
print(cand.head(10)[["name","district","predicted_speed","prediction_std",
                     "straddles","pop_2km","measurement_priority"]].to_string(index=False))
print("\n⚠ Ranks WHERE TO COLLECT DATA. NOT an infrastructure recommendation.")

unmeasured settlements        : 216
with population within 2 km   : 111
straddling the 21 Mbps line   : 14

gate check — top 14 are all straddlers: 14/14
correlation with model disagreement: -0.356

TOP 10 — inspect these by eye before anyone acts on them
               name district  predicted_speed  prediction_std  straddles  pop_2km  measurement_priority
           Binakaan Keningau        25.740000        6.500000       True      574                 0.918
          Bangawong Keningau        23.000000        3.670000       True      435                 0.912
  Kampung Baru-Baru   Tuaran        31.030001       11.110000       True      550                 0.908
       Namadan Baru Keningau        23.049999        4.470000       True      213                 0.895
             Katubu    Tenom        17.080000        6.110000       True      194                 0.885
         Tiga Tarok    Pitas        38.810001       18.969999       True      245                 0.878
Kampung Pamunter

---
## 3. Survey routing

The decisive candidates are scattered across districts. A team based in one region cannot
chase priority order across the state, so routing is done **per district** — the unit a field
team actually works in.

**This is classical optimisation, not machine learning.** Greedy nearest-neighbour to build a
tour, then 2-opt to remove crossings. At this number of stops it runs in milliseconds and the
result is near-optimal.

⚠️ **Straight-line distance.** There is no road network here, so every figure is a **lower
bound** on real driving. Sabah's mountains and rivers mean actual distances are considerably
longer. The output is a sensible visiting order, not a navigation plan.

In [7]:
# ══════════════════════════════════════════════════════════════════════
# SURVEY ROUTING — per district, greedy nearest-neighbour + 2-opt.
# Districts with a single candidate are listed, not routed: a one-stop
# "route" is not a route.
# ══════════════════════════════════════════════════════════════════════
def hav(a, b):
    """Great-circle distance in km between two (lat, lon) pairs."""
    la1, lo1, la2, lo2 = map(np.radians, [a[0], a[1], b[0], b[1]])
    h = np.sin((la2-la1)/2)**2 + np.cos(la1)*np.cos(la2)*np.sin((lo2-lo1)/2)**2
    return 2 * R_KM * np.arcsin(np.sqrt(h))

def route(pts):
    n = len(pts)
    if n < 2: return list(range(n)), 0.0
    Dm = np.array([[hav(p, q) for q in pts] for p in pts])
    tour, unv = [0], set(range(1, n))            # start at the highest-priority stop
    while unv:                                    # greedy nearest-neighbour
        j = min(unv, key=lambda k: Dm[tour[-1], k]); tour.append(j); unv.discard(j)
    improved = True                               # 2-opt: reverse segments that cross
    while improved:
        improved = False
        for i in range(1, n-1):
            for k in range(i+1, n):
                if Dm[tour[i-1],tour[i]] + Dm[tour[k],tour[(k+1)%n]] > \
                   Dm[tour[i-1],tour[k]] + Dm[tour[i],tour[(k+1)%n]]:
                    tour[i:k+1] = tour[i:k+1][::-1]; improved = True
    return tour, sum(Dm[tour[i], tour[i+1]] for i in range(n-1))

top = cand[cand.straddles].copy()
print(f"routing {len(top)} decisive candidates across {top.district.nunique()} districts\n")

rows, total = [], 0.0
multi, single = [], []
for dist, g in top.groupby("district"):
    (single if len(g) < 2 else multi).append((dist, g))

for dist, g in sorted(multi, key=lambda x: -len(x[1])):
    g = g.sort_values("measurement_priority", ascending=False).reset_index(drop=True)
    tour, km = route(list(zip(g.lat, g.lon)))
    total += km
    print(f"── {dist}: {len(g)} stops, {km:.0f} km ──")
    for order, i in enumerate(tour, 1):
        r = g.loc[i]
        print(f"  {order}. {str(r['name'])[:30]:30s} est {r.predicted_speed:5.1f} "
              f"±{r.prediction_std:4.1f}  {int(r.pop_2km):5d} people")
        rows.append({"district":dist, "stop_order":order, "settlement_id":r.settlement_id,
                     "name":r["name"], "lat":r.lat, "lon":r.lon,
                     "predicted_speed":r.predicted_speed, "prediction_std":r.prediction_std,
                     "pop_2km":r.pop_2km, "route_km_district":round(km,1)})
    print()

if single:
    print("── Single-candidate districts (visit individually, no route) ──")
    for dist, g in single:
        r = g.iloc[0]
        print(f"  {dist:14s} {str(r['name'])[:30]:30s} est {r.predicted_speed:5.1f} "
              f"±{r.prediction_std:4.1f}  {int(r.pop_2km):5d} people")
        rows.append({"district":dist, "stop_order":1, "settlement_id":r.settlement_id,
                     "name":r["name"], "lat":r.lat, "lon":r.lon,
                     "predicted_speed":r.predicted_speed, "prediction_std":r.prediction_std,
                     "pop_2km":r.pop_2km, "route_km_district":0.0})

pd.DataFrame(rows).to_csv(D + "outputs/survey_routes_v1.csv", index=False)

pts   = list(zip(top.lat, top.lon))
naive = sum(hav(p, q) for p, q in zip(pts[:-1], pts[1:]))
print(f"\nvisiting in raw priority order : {naive:6.0f} km")
print(f"routed per district            : {total:6.0f} km  ({naive/max(total,1):.1f}x less travel)")
print("\n⚠ Straight-line, no road network — a LOWER BOUND on real driving distance.")
print("saved ✓ outputs/survey_routes_v1.csv")

routing 14 decisive candidates across 7 districts

── Keningau: 4 stops, 35 km ──
  1. Binakaan                       est  25.7 ± 6.5    574 people
  2. Bangawong                      est  23.0 ± 3.7    435 people
  3. Namadan Baru                   est  23.0 ± 4.5    213 people
  4. Kaliwot                        est  25.7 ±11.0     88 people

── Tuaran: 3 stops, 46 km ──
  1. Kampung Baru-Baru              est  31.0 ±11.1    550 people
  2. Kampung Paka Kiulu             est  26.6 ± 7.0     22 people
  3. Kampung Kotunuan Lama          est  19.6 ± 4.8     74 people

── Tambunan: 2 stops, 2 km ──
  1. Kitoring                       est  19.6 ± 9.8     23 people
  2. Monsok Tengah                  est  33.3 ±13.7     23 people

── Tenom: 2 stops, 2 km ──
  1. Katubu                         est  17.1 ± 6.1    194 people
  2. Ulu Patian                     est  22.4 ± 7.2     86 people

── Single-candidate districts (visit individually, no route) ──
  Beluran        Dolamos              

---
## 4. Line-of-sight terrain screen

One mast can serve several villages, so costing one mast per settlement overstates the bill.
But a distance-only estimate assumes every path is clear — and Sabah is mountainous:

```
mast ●─────────────● village     flat: signal gets through
mast ●────╱▔▔╲─────● village     ridge in the way: blocked
```

Both are the same distance. Only one works.

This section samples **350 elevation points along every mast-to-village path** and tests
whether the ground rises into the signal.

### Three corrections a naive sightline misses

| Correction | Why |
|---|---|
| **Earth curvature (4/3 model)** | over 10 km the planet itself bulges into the path; the 4/3 factor also approximates atmospheric refraction |
| **Fresnel zone** | radio needs clearance *around* the line, not just a bare sightline. 60% of the first Fresnel radius is the standard criterion, and it is stricter than line-of-sight |
| **`min_clear_m`** | how much room there was, so failing by 2 m and failing by 200 m stay distinguishable |

### Assumptions

- **Mast height 30 m** — the height the per-site cost already assumes.
- **Receiver height 10 m** — ITU-R P.1546 reference receiving height.
- **700 MHz** — Malaysia's assigned sub-1 GHz band.

⚠️ **This is a terrain screen, not predicted coverage.** SRTM is C-band radar, so in dense
forest the reading sits partway up the canopy rather than at ground level. It therefore
over-blocks cleared land and under-blocks tall forest, and cannot be corrected without a
canopy-height layer. Real propagation needs ITU-R P.1812 profiles.

In [8]:
# ══════════════════════════════════════════════════════════════════════
# Rebuild the Sabah elevation raster (~419 MB) and cache it to Drive so
# this only happens once.
#
# gdal-bin MUST be installed before eio runs, or the merge fails silently
# and writes a 0-byte file — after which every sample returns NaN with no
# error raised.
# ══════════════════════════════════════════════════════════════════════
RAST = D + "srtm_sabah.tif"

if os.path.exists(RAST) and os.path.getsize(RAST) > 1e8:
    !cp "{RAST}" srtm_sabah.tif
    print("restored from Drive ✓")
else:
    !apt-get install -y gdal-bin -q
    !pip install elevation -q
    for name, w, s, e, n in [("a",115.0,4.0,117.35,5.8), ("b",117.35,4.0,119.7,5.8),
                             ("c",115.0,5.8,117.35,7.6), ("d",117.35,5.8,119.7,7.6)]:
        print("chunk", name, "...")
        !eio clip -o srtm_{name}.tif --bounds {w} {s} {e} {n}
    !gdal_merge.py -o srtm_sabah.tif srtm_a.tif srtm_b.tif srtm_c.tif srtm_d.tif
    !cp srtm_sabah.tif "{RAST}"
    print("built and cached to Drive ✓")

!ls -lh srtm_sabah.tif      # ~419 MB. If 0, gdal-bin did not install.

restored from Drive ✓
-rw------- 1 root root 419M Aug  9 16:27 srtm_sabah.tif


In [9]:
# ══════════════════════════════════════════════════════════════════════
# TERRAIN LINE-OF-SIGHT SCREEN
#
# For each path: interpolate 350 points from mast to village, sample the
# elevation at each, and compare the ground profile against the straight
# line from mast-top to receiver-top after adding the earth bulge.
#
# A void at either endpoint makes a path UNKNOWN, not blocked. Without
# this, np.nanmin returns NaN, NaN > 0 evaluates False, and missing data
# would silently be reported as terrain blockage.
# ══════════════════════════════════════════════════════════════════════
import rasterio

N      = 350            # 29 m spacing on a 10 km path; the SRTM cell is 30 m
H_MAST = 30.0
H_RECV = 10.0            # ITU-R P.1546 reference receiving height
K_EARTH, R_E = 4/3, 6371000.0
LAMBDA = 3e8 / 700e6    # 700 MHz

pairs = pd.read_csv(D + "tower_pairs.csv")
print(f"{len(pairs)} mast→village paths")
print(pairs.radius_km.value_counts().sort_index().to_string(), "\n")

frac = np.linspace(0, 1, N)
lat = pairs.mast_lat.values[:,None] + (pairs.s_lat - pairs.mast_lat).values[:,None] * frac
lon = pairs.mast_lon.values[:,None] + (pairs.s_lon - pairs.mast_lon).values[:,None] * frac

with rasterio.open("srtm_sabah.tif") as src:
    nod = src.nodata if src.nodata is not None else -32768     # never trust None
    z = np.fromiter((v[0] for v in src.sample(zip(lon.ravel(), lat.ravel()))),
                    float, count=lat.size)
z = z.reshape(lat.shape)
z[(z == nod) | (z < -100) | (z > 5000)] = np.nan               # voids and absurd values

d_tot = pairs.dist_km.values[:,None] * 1000.0
dist  = frac * d_tot
bulge = dist * (d_tot - dist) / (2 * K_EARTH * R_E)            # 4/3-earth curvature
sight = (z[:,:1] + H_MAST) + ((z[:,-1:] + H_RECV) - (z[:,:1] + H_MAST)) * frac
clear = sight - (z + bulge)                                    # +ve = path is above ground
r1    = np.sqrt(LAMBDA * dist * (d_tot - dist) / d_tot)        # first Fresnel radius

mid   = slice(1, -1)                                           # endpoints are trivially clear
mc    = np.nanmin(clear[:, mid], axis=1)
fc    = np.nanmin((clear - 0.6*r1)[:, mid], axis=1)            # 60% Fresnel criterion
voids = np.isnan(z).sum(axis=1)
bad   = np.isnan(z[:,0]) | np.isnan(z[:,-1]) | np.isnan(mc)    # UNKNOWN, not blocked

pairs["min_clear_m"] = np.where(bad, np.nan, mc.round(1))
pairs["los"]         = np.where(bad, pd.NA, mc > 0)
pairs["fresnel"]     = np.where(bad, pd.NA, fc > 0)
pairs["voids"]       = voids
pairs["usable"]      = ~bad

print("pass rate by assumed radius (usable paths only):")
print(pairs[pairs.usable].groupby("radius_km")[["los","fresnel"]].mean().round(3).to_string())
print("\nSanity: fresnel must be BELOW los at every radius (it is the stricter test),")
print("and pass rates must fall as radius grows (longer paths cross more terrain).")
print(f"\nunusable (endpoint void) : {int(bad.sum())} of {len(pairs)}")
print(f"paths with any void      : {int((voids > 0).sum())}")
print(f"min_clear_m quartiles    : {np.nanpercentile(pairs.min_clear_m,[25,50,75]).round(1)}")

pairs.to_csv(D + "outputs/tower_pairs_los.csv", index=False)
print("\nsaved ✓ outputs/tower_pairs_los.csv")

10070 mast→village paths
radius_km
3     1412
5     2596
10    6062 

pass rate by assumed radius (usable paths only):
                los   fresnel
radius_km                    
3          0.623229  0.504249
5          0.508475   0.36171
10         0.331739  0.190861

Sanity: fresnel must be BELOW los at every radius (it is the stricter test),
and pass rates must fall as radius grows (longer paths cross more terrain).

unusable (endpoint void) : 0 of 10070
paths with any void      : 0
min_clear_m quartiles    : [-61.9  -9.    7.7]

saved ✓ outputs/tower_pairs_los.csv


---
## 5. Nighttime lights — an electrification constraint

A mast needs power. A site with no grid connection needs solar or a generator, which is a
different cost class.

VIIRS nighttime lights measure electrification and human activity, so they answer that
question — and nothing else.

### 🚫 The rule this must not break

**Nightlights do not measure connectivity.** If "dark" is ever rendered or read as "no
coverage", it breaks the principle the whole project rests on: *missing data is uncertainty,
never poor service*. A dark settlement may have perfectly good mobile service; it has no grid
electricity, which is a separate problem.

Permitted: power availability, electrification context, demand proxy.
**Not permitted: coverage, service quality, or any input to DIPI.**

### Two method choices

**Annual composites, not monthly.** The annual VIIRS product applies outlier removal to
discard biomass-burning pixels using the twelve-month median. In Sabah that filters out palm-oil
burning and gas flares at source — exactly the contamination that would otherwise read as
"brightening settlements". Raw monthly imagery has no such filtering.

**Maximum within 1 km, not mean over 2 km.** A 2 km buffer at 500 m resolution is about 50
pixels. A village lighting one or two of them has its signal divided by fifty, so a small
kampung on mains power reads as dark. The maximum is dilution-free.

In [10]:
# ══════════════════════════════════════════════════════════════════════
# Earth Engine authentication.
# VIIRS is only published as global rasters, so downloading twelve years
# to extract Sabah is impractical. Earth Engine clips server-side and
# returns a small table — the only reason it is used in this project.
# ══════════════════════════════════════════════════════════════════════
import ee
ee.Authenticate()
ee.Initialize(project="dino-slayer")        # replace with your own Cloud project id

for c in ["NOAA/VIIRS/DNB/ANNUAL_V21", "NOAA/VIIRS/DNB/ANNUAL_V22"]:
    ic  = ee.ImageCollection(c)
    ids = ic.aggregate_array("system:index").getInfo()
    print(f"{c}: {len(ids)} images → {ids}")

NOAA/VIIRS/DNB/ANNUAL_V21: 9 images → ['20130101', '20140101', '20150101', '20160101', '20170101', '20180101', '20190101', '20200101', '20210101']
NOAA/VIIRS/DNB/ANNUAL_V22: 4 images → ['20220101', '20230101', '20240101', '20250101']


In [11]:
# ══════════════════════════════════════════════════════════════════════
# Extract annual radiance per settlement: MAX within 1 km.
# V21 covers 2013-2021, V22 covers 2022 onward.
#
# The two are different processing versions, so a step change at the
# boundary would read as a state-wide brightening trend. The check for
# that is in the next cell.
#
# Requests are chunked because getInfo has a payload limit.
# ══════════════════════════════════════════════════════════════════════
import time

pts = [ee.Feature(ee.Geometry.Point([r.lon, r.lat]).buffer(1000), {"sid": r.settlement_id})
       for r in df.itertuples()]
RED = ee.Reducer.max().combine(ee.Reducer.mean(), sharedInputs=True)

rows = []
for yr in range(2013, 2026):
    coll = "ANNUAL_V22" if yr >= 2022 else "ANNUAL_V21"
    img  = ee.Image(f"NOAA/VIIRS/DNB/{coll}/{yr}0101").select("average_masked")
    for i in range(0, len(pts), 200):
        r = img.reduceRegions(ee.FeatureCollection(pts[i:i+200]), RED, 500).getInfo()
        rows += [{"settlement_id": ft["properties"]["sid"], "year": yr,
                  "max_1km":  ft["properties"].get("max"),
                  "mean_1km": ft["properties"].get("mean")} for ft in r["features"]]
        time.sleep(0.5)
    print(f"{yr} done")

ntl = pd.DataFrame(rows)
ntl.to_csv(D + "outputs/viirs_annual_max1km.csv", index=False)
print(f"\nsaved ✓ {len(ntl)} rows = {df.shape[0]} settlements × 13 years")

piv = ntl.pivot(index="settlement_id", columns="year", values="max_1km")
print("\nmedian radiance by year — check for a STEP at the 2021/2022 boundary,")
print("which would be a processing-version artefact, not a real trend:")
print(piv.median().round(3).to_string())

2013 done
2014 done
2015 done
2016 done
2017 done
2018 done
2019 done
2020 done
2021 done
2022 done
2023 done
2024 done
2025 done

saved ✓ 18824 rows = 1448 settlements × 13 years

median radiance by year — check for a STEP at the 2021/2022 boundary,
which would be a processing-version artefact, not a real trend:
year
2013    0.0
2014    0.0
2015    0.0
2016    0.0
2017    0.0
2018    0.0
2019    0.0
2020    0.0
2021    0.0
2022    0.0
2023    0.0
2024    0.0
2025    0.0


In [12]:
# ══════════════════════════════════════════════════════════════════════
# TREND per settlement.
#
# Mann-Kendall (via Kendall's tau) tests for a monotonic trend without
# assuming linearity or normality; Theil-Sen gives a slope resistant to
# outliers. Both are appropriate for short, noisy environmental series.
#
# The urban-artefact reclassification is applied BEFORE saving: VIIRS DNB
# cannot sense blue light, so a city converting sodium street lamps to
# white LEDs reads as falling radiance even when lighting increases. That
# shows up only in the brightest places, so bright "dimming" settlements
# are relabelled rather than reported as a finding.
# ══════════════════════════════════════════════════════════════════════
from scipy.stats import kendalltau, theilslopes

LIT_THRESHOLD = 0.5     # tested at 0.5 / 0.25 / 0.1 — the detection floor, not the threshold, is the constraint (745 / 798 / 804 lit)
yrs = piv.columns.values.astype(float)

res = []
for sid, row in piv.iterrows():
    v = row.values.astype(float)
    if np.nanmax(v) < LIT_THRESHOLD:                  # no detectable light in 13 years
        res.append({"settlement_id": sid, "ever_lit": False, "light_trend": "dark",
                    "ntl_max": round(float(np.nanmax(v)), 3)})
        continue
    tau, p = kendalltau(yrs, v)
    slope  = theilslopes(v, yrs)[0]
    res.append({"settlement_id": sid, "ever_lit": True,
                "ntl_max":   round(float(np.nanmax(v)), 3),
                "ntl_2025":  round(float(v[-1]), 3),
                "ntl_slope": round(float(slope), 4),
                "ntl_p":     round(float(p), 4),
                "light_trend": "brightening" if p < 0.05 and slope > 0
                          else "dimming"     if p < 0.05 and slope < 0
                          else "flat"})

t = pd.DataFrame(res)

# Relabel bright "dimming" as an LED-conversion artefact, THEN save.
t["light_trend"] = np.where((t.light_trend == "dimming") & (t.ntl_max > 10),
                            "flat_urban_artefact", t.light_trend)
t.to_csv(D + "outputs/viirs_trend.csv", index=False)

print(t.light_trend.value_counts().to_string())
print(f"\nlit : {int(t.ever_lit.sum())} of {len(t)} ({t.ever_lit.mean()*100:.0f}%)")
print(f"dark: {int((~t.ever_lit).sum())}")

# Is the threshold load-bearing, or is the detection floor doing the work?
mx = piv.max(axis=1)
print("\nlit count by threshold — if these are close, the threshold does not matter:")
for th in [0.5, 0.25, 0.1]:
    print(f"  >= {th:<5} {int((mx >= th).sum()):5d}")
print("0.1 is near the VIIRS noise floor, so settlements dark at 0.1 are dark, full stop.")

light_trend
dark                   703
flat                   422
brightening            276
dimming                 31
flat_urban_artefact     16

lit : 745 of 1448 (51%)
dark: 703

lit count by threshold — if these are close, the threshold does not matter:
  >= 0.5     745
  >= 0.25    798
  >= 0.1     804
0.1 is near the VIIRS noise floor, so settlements dark at 0.1 are dark, full stop.


In [19]:
# ══════════════════════════════════════════════════════════════════════
# THE JOIN THAT PAYS OFF — own mast AND own power.
#
# Neither dataset shows this alone: the isolation flags say which tower
# settlements nothing else can reach, and VIIRS says which have no grid.
# Settlements in both sets are the most expensive to serve in Sabah.
# ══════════════════════════════════════════════════════════════════════
iso = pd.read_csv(D + "tower_isolated.csv")
lit = (mx >= LIT_THRESHOLD).rename("ever_lit").reset_index()
j   = iso.merge(lit, on="settlement_id", how="left")

for r in [3, 5, 10]:
    m = j[f"only_self_{r}km"] == 1
    print(f"{r:>2} km assumed reach: {m.sum():3d} isolated · "
          f"{int((m & ~j.ever_lit).sum()):3d} of those also unlit → own mast AND own power")

j.to_csv(D + "outputs/tower_isolated_power.csv", index=False)
print("\nsaved ✓ outputs/tower_isolated_power.csv")
print("\nThis count barely moves across the three radii, so unlike most tower")
print("figures it does not depend on the assumed reach radius.")

 3 km assumed reach: 191 isolated ·  67 of those also unlit → own mast AND own power
 5 km assumed reach: 169 isolated ·  58 of those also unlit → own mast AND own power
10 km assumed reach: 157 isolated ·  56 of those also unlit → own mast AND own power

saved ✓ outputs/tower_isolated_power.csv

This count barely moves across the three radii, so unlike most tower
figures it does not depend on the assumed reach radius.


---
## Notebook 3 complete

### Outputs in `MyDrive/Dataset/outputs/`

| File | Contents |
|---|---|
| `clusters.json` | bundle id and shared trunk length per settlement |
| `cluster_sensitivity.json` | the `min_cluster_size` sweep |
| `measurement_priority_v1.csv` | unmeasured settlements ranked by decision relevance |
| `survey_routes_v1.csv` | per-district visiting order for the decisive candidates |
| `tower_pairs_los.csv` | terrain screen per mast→village path |
| `viirs_annual_max1km.csv` | 13-year radiance series per settlement |
| `viirs_trend.csv` | electrification trend class per settlement |
| `tower_isolated_power.csv` | isolated **and** unlit — the most expensive to serve |

---

### What each section may and may not claim

| Output | May say | May **not** say |
|---|---|---|
| Bundles | these communities could share one build; the shared trunk is shorter than individual runs | this is an engineering design, or a driving route |
| Measurement priority | measuring here would most change a decision | this community is underserved |
| Survey routes | a sensible visiting order; straight-line lower bound | actual driving distance or time |
| LoS screen | terrain blocks this path under stated assumptions | predicted radio coverage |
| VIIRS | no detectable electrification, so power must be supplied | no mobile coverage |

---